In [1]:
import os

base = '/kaggle/input/datasets/duydieunguyen/licenseplates'
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if files:
            print(f'{indent}  ({len(files)} files) — ví dụ: {files[0]}')
    

licenseplates/
  (1 files) — ví dụ: dataset.yaml
  labels/
    val/
      (1145 files) — ví dụ: Hung_0213.txt
    train/
      (3433 files) — ví dụ: greenpack_0267.txt
  images/
    val/
      (1145 files) — ví dụ: Tgmt_0816.png
    train/
      (3433 files) — ví dụ: Tgmt_0441.png


In [2]:
with open('/kaggle/input/datasets/duydieunguyen/licenseplates/dataset.yaml', 'r') as f:
    print(f.read())

train: TongHop\YOLODataset/images/train/
val: TongHop\YOLODataset/images/val/
test: TongHop\YOLODataset/images/test/
nc: 2
names: ['BSD', 'BSV']


In [3]:
import yaml
import os

# Ghi lại yaml với đường dẫn đúng trên Kaggle
data = {
    'train': '/kaggle/input/datasets/duydieunguyen/licenseplates/images/train',
    'val': '/kaggle/input/datasets/duydieunguyen/licenseplates/images/val',
    'nc': 2,
    'names': ['BSD', 'BSV']
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f)

print("Đã tạo data.yaml mới")

# Kiểm tra số ảnh
for split in ['train', 'val']:
    path = f'/kaggle/input/datasets/duydieunguyen/licenseplates/images/{split}'
    n = len(os.listdir(path))
    print(f"{split}: {n} ảnh")

# Xem thử 1 label để hiểu format
label_path = '/kaggle/input/datasets/duydieunguyen/licenseplates/labels/train'
sample = os.listdir(label_path)[0]
with open(f'{label_path}/{sample}') as f:
    print(f"\nSample label ({sample}):")
    print(f.read())

Đã tạo data.yaml mới
train: 3433 ảnh
val: 1145 ảnh

Sample label (greenpack_0267.txt):
1 0.45307662538699695 0.5354747162022704 0.6765673374613004 0.5535345717234262 0.6649574303405573 0.8076625386996904 0.44473684210526315 0.8



In [4]:
!pip install ultralytics -q
from ultralytics import YOLO

# Dùng yolov8s-obb cho oriented bounding box
model = YOLO('yolov8s-obb.pt')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,          # Kaggle GPU mạnh hơn, batch 16 được
    device=0,
    project='/kaggle/working/runs',
    name='license_plate_obb',
    patience=10,
    save=True,
    plots=True
)

print("Train xong!")
print(f"Best mAP50: {results.results_dict['metrics/mAP50(B)']:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [5]:
import shutil

shutil.copy('/kaggle/working/runs/license_plate_obb/weights/best.pt',
            '/kaggle/working/best.pt')
print("Sẵn sàng download!")

Sẵn sàng download!


In [6]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/license_plate_obb/weights/best.pt')
results = model.predict(
    source='/kaggle/input/datasets/duydieunguyen/licenseplates/images/val',
    conf=0.5,
    save=True,
    project='/kaggle/working',
    name='predict_test'
)
print("Xem ảnh kết quả tại /kaggle/working/predict_test/")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1145 /kaggle/input/datasets/duydieunguyen/licenseplates/images/val/Dieu_0003.png: 480x640 2 BSVs, 59.3ms
image 2/1145 /kaggle/input/datasets/duydieunguyen/licenseplates/images/val/Dieu_0005.png: 480x640 1 BSV, 13.6ms
image 3/1145 /kaggle/input/datasets/duydieunguyen/licenseplates/images/val/Dieu_0017.png: 480x640 1 BSV, 13.6ms
image 4/1145 /kaggle/input/datasets/duydieunguyen/licenseplates/images/val/Dieu_0018.png: 480x640 2 BSVs, 13.6ms
image 5/